### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="coffee_rating_prediction",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/hanifalirsyad/coffee-scrap-coffeereview",
    download_description="""
We download the data from Kaggle, which comes from scrapping www.coffeereview.com.

kaggle datasets download hanifalirsyad/coffee-scrap-coffeereview -f coffee_clean.csv && unzip coffee_clean.csv.zip && rm coffee_clean.csv.zip
mkdir -p local-data-warehouse/coffee_rating_prediction && mv coffee_clean.csv local-data-warehouse/coffee_rating_prediction
""",
    # References
    academic_reference_bibtex=r"""@misc{AlIrsyad2023CoffeeDataCoffeeReview,
  author = {Hanif Al Irsyad},
  title  = {Coffee Data CoffeeReview},
  year   = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/hanifalirsyad/coffee-scrap-coffeereview}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="AlIrsyad2023CoffeeDataCoffeeReview",
    license="None",
    data_tags=["Non-IID", "Temporal", "Spatial", "2ndTierData"],
    curation_comments="""
We start with the Kaggle version. Note, we acknowledge that the dataset is not of the highest quality and likely does not represent a real-world prediction task. Nevertheless, it can be used to test models.

- We drop slug as it is a unique identifier that does not have predictive power.
- We drop desc_2 as it in general includes notes that may or may not be related to the coffee quality. Moreover, it sometimes leaks the rating (e.g. through award notes). We only keep desc_1 and desc_3, which are mostly descriptions of the coffee.
- We drop all_text as it is a concatenation of desc_1, desc_2, and desc_3. And desc_2 includes data leakage (such as if the coffee won an award or not). Thus, we drop all_test.
- We drop aroma, acid, body, flavor, aftertaste, and with_milk as they are all features of the rating and thus leaking the target. Any of them could be used as a target, we focus on the overall rating.
- We parse agtron into a lower and upper value as float.
- Note, the data contains several spatial features about locations. We do not resolve these.
- The price feature is extremely messy as a result of web scrapping. We resolve this feature as real tabular data (not from a web scrapper but from the database source), would have these information stored in a more structured way. We drop coffees that are measured in "capsules" units as they exist only 8 times and seem to be strong outliers. We normalizes the price to be per grams. Moreover, we convert all currencies to USD (with a fixed rate); we ignore the temporal incorrectness this may introduce as it is most likely minimal.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="rating",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="review_date",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "coffee_clean.csv")
print("Loaded data shape:", df.shape)

# Ensure date time
df["review_date"] = pd.to_datetime(df["review_date"], format="%B %Y")


# Parse argton
df[["agtron_lower", "agtron_upper"]] = (
    df["agtron"]
    .str.split("/", expand=True)
)
df["agtron_lower"] = df["agtron_lower"].replace(
    {
        "": np.nan, "NA": np.nan,
         # Fix parsing errors
         "5252": "52", "547": "54", "555": "55",
     }
).astype(float)
df["agtron_upper"] = df["agtron_upper"].replace({"": np.nan, "NA": np.nan}).astype(float)


# Parse est_price
s = df["est_price"].astype("string")
# --- normalize a bit (handles 'NT$650/230g', extra spaces, etc.)
norm = (
    s.str.strip()
     .str.replace(r"\s+", " ", regex=True)
     .str.replace(r"NT\s*\$", "NT $", regex=True)    # NT$ -> NT $
     .str.replace(r"\$(?=\d)", "$", regex=True)      # keep $ tight to digits ok
)
# --- flag messy rows (parentheses, multiple prices separated by ';', or extra explanatory text)
messy_pat = r"\(|\)|;|currently on sale|limited availability|includes shipping|available in store only|see website|more information"
is_messy = norm.str.contains(messy_pat, case=False, regex=True)
norm = norm.replace({
    # Handle all special cases
    "NA (available in store only)": pd.NA,
    "$32.00/boxed set (125 grams of this coffee plus 125 grams of an anaerobic-processed coffee from the same farm)": "$32.00/125 grams",
    "$280.00/70 grams; $200.00/50 grams": "$280.00/70 grams",
    "$4.00/12-ounce bottle; $20.00/6-pack": "$4.00/12-ounce bottle",
    "$18.00/4-12-ounce bottles; 32 ounces/$12; 64-ounces/$20.00": "$18.00/4-12-ounce bottles",
    "$3.00/sachet (plus one donated)": "$3.00/sachet",
    "$15.00/20 ounces (2 types)": "$15.00/20 ounces",
    "$45.95/8 ounces (currently on sale for $36.76)": "$45.95/8 ounces",
     "$15.00/12 ounces; $35.00/2 pounds": "$15.00/12 ounces",
     "$13.99/12 ounces ($79.00/5 pounds)": "$13.99/12 ounces",
     "$21.00/12 ounces (includes shipping)": "$21.00/12 ounces",
     "$16.00/12 ounces; $50.00/5 pounds": "$16.00/12 ounces",
     "€29.95/1 kilo (35.3 ounces)": "€29.95/1 kilo",
     "$125.00/4 ounces; limited availability": "$125.00/4 ounces",
     '$39.95/8 ounces (packaged as a "duo" with Bourbon Rey Guatemala)': '$39.95/8 ounces',
     '$39.95/8 ounces (packaged as a "duo" with the Bourbon Rey Jamaica)': '$39.95/8 ounces',
     "See website for more information": pd.NA,
})
pat_clean = (
    r"^\s*(?:(?P<special>NT)\s*)?"
    r"(?P<currency>(?:USD|US|CAD|AUD|HKD|HK|AED|RMB|RM|IDR|KRW|NTD)\s*\$?|[\$€£¥])\s*"
    r"(?P<price>\d{1,3}(?:,\d{3})*(?:\.\d+)?|\d+(?:\.\d+)?)\s*/\s*"
    r"(?P<unit_qty>\d+(?:\.\d+)?)\s*"
    r"(?P<unit>ounces?|oz|grams?|g|ml|kilo|kg|pounds?|lb|capsules?|caps)\.?\s*$"
)
parsed = norm.str.extract(pat_clean)
df["NT_price"] = parsed["special"]
df["currency_raw"] = parsed["currency"]
df["price"] = (
    parsed["price"].str.replace(",", "", regex=False).astype("float")
)
df["unit_qty"] = parsed["unit_qty"].astype("float")
df["unit_raw"] = parsed["unit"].str.lower()

# unify qty
to_grams = {
    "g": 1,
    "grams": 1,
    "pounds": 453.59237,
    "kilo": 1000,
    "ounces": 28.349523125,
}

mask = df["unit_raw"].isin(list(to_grams.keys()))

df.loc[mask, "unit_qty"] = (
    df.loc[mask, "unit_qty"] * df.loc[mask, "unit_raw"].map(to_grams)
)

df.loc[mask, "unit_raw"] = "grams"
df = df[df["unit_raw"] != "capsules"].reset_index(drop=True)
df["price_per_gram"] = df["price"] / df["unit_qty"]
fx_to_usd = {
    "HKD $": 0.128,
    "HK $": 0.128,
    "CAD $": 0.74,
    "AUD $": 0.66,
    "USD $": 1.0,
    "US $": 1.0,
    "¥": 0.0067,
    "RMB": 0.14,
    "RMB ": 0.14,
    "£": 1.27,
    "€": 1.09,
    "NTD $": 0.031,
    "RM": 0.21,
    "RM ": 0.21,
    "AED $": 0.27,
    "IDR $": 0.000064,
    "KRW $": 0.00075,
    "$": 1.0,
}
assert set(df["currency_raw"].dropna().unique()).issubset(set(fx_to_usd.keys())), f"Found currencies not in fx_to_usd mapping: {set(df['currency_raw'].dropna().unique()) - (set(fx_to_usd.keys()))}"
df["price_per_gram_in_usd"] = df["price_per_gram"] * df["currency_raw"].map(fx_to_usd)
df = df.drop(columns=["unit_qty", "unit_raw", "price", "currency_raw", "price_per_gram"])

as_cat_column = [
    "roast",
    "NT_price"
]
as_str_column = [
    "roaster",
    "name",
    "location",
    "origin",
    "desc_1",
    "desc_3",
]
for c in as_str_column:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_column] = df[as_cat_column].astype("category")

# Drop columns
df = df.drop(columns=["slug", "desc_2", "all_text", "aroma", "acid", "body", "flavor", "aftertaste", "with_milk", "agtron", "est_price"])
# drop the 1 duplicate row
df = df.drop_duplicates()
df = df.sort_values(by="review_date").reset_index(drop=True)

Loaded data shape: (2440, 20)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,369
Columns: 13
Use sampling: False (sample size: 2,369)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['desc_3', 'desc_1', 'name', 'origin', 'price_per_gram_in_usd', 'roaster', 'location', 'review_date', 'agtron_upper', 'agtron_lower']
Rows remaining as candidates after top-10 filter: 0 (of 2,369)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,rating,roaster,name,location,origin,roast,review_date,desc_1,desc_3,agtron_lower,agtron_upper,NT_price,price_per_gram_in_usd
0,93,Caffeic,Ethiopia,"Helena, Montana","Bench-Maji Zone, southern Ethiopia",Medium-Light,2018-10-01,"Crisply sweet, fruit-toned. Raspberry, roasted cacao nib, honey, fresh-cut fir, freesia-like flowers in aroma and cup. Richly sweet-tart in structure with vibrant acidity; delicate, silky mouthfeel. The long, lasting finish centers around cocoa-toned honey and freesia.",A sweetly fruit-forward natural-processed Ethiopia with spicy-floral intrigue.Explore Similar CoffeesClick here for more reviews from CaffeicClick here for more information about coffees from Ethiopia,56.0,82.0,<NA>,0.039683
1,95,Branch Street Coffee Roasters,Kenya Konyu,"Youngstown, Ohio","Kirinyaga District, south-central Kenya",Medium-Light,2018-10-01,"Vibrant, rich-toned, sweetly-tart. Tart berry, roasted cacao nib, crisp, bittersweet hop flowers, sandalwood, agave syrup in aroma and cup. Sweetly tart structure with engaging, juicy acidity; syrupy-smooth mouthfeel. The savory-leaning finish is driven by hop flowers, with cocoa nib as another throughline supporting the dried fruit and spicy aromatic wood notes.","A multi-layered Kenya coffee with a perfect balance of sweet, tart and savory notes — in particular, dried berries and richly bittersweet flowers.Explore Similar CoffeesClick here for more reviews from Branch Street Coffee RoastersClick here for more information about coffees from Kenya",57.0,81.0,<NA>,0.057320
2,92,Magnolia Coffee,Guatemala Finca San Gerardo,"Charlotte, North Carolina","Lake Amatitlán area, south-central Guatemala",Medium-Light,2018-10-01,"Crisp, richly sweet. Magnolia, dried persimmon, hazelnut, molasses, cedar in aroma and cup. Sweet in structure with round, gentle acidity; satiny-smooth mouthfeel. The finish is resonant with magnolia, dried persimmon and hazelnut notes.","A friendly, floral-toned Guatemala cup with undercurrents of rich hazelnut.Explore Similar CoffeesClick here for more reviews from Magnolia CoffeeClick here for more information about coffees from GuatemalaVisit Magnolia Coffee",52.0,72.0,<NA>,0.044092
3,95,Dragonfly Coffee Roasters,Hacienda La Esmeralda Cabana Geisha Natural,"Boulder, Colorado","Boquete growing region, western Panama",Medium-Light,2018-10-01,"Rich, resonant, high-toned. Lychee, lilac, chocolate fudge, frankinense, tamarind in aroma and cup. Sweet-tart structure with juicy, bright acidity; plush, syrupy-smoooth mouthfeel. The lingering finish is flavor-saturated, making good on the promise of the cup’s deep sweetness.","A complex, multi-layered, exquisitely clean natural-processed Geisha cup.Explore Similar CoffeesClick here for more reviews from Dragonfly Coffee RoastersClick here for more information about coffees from Panama",57.0,83.0,<NA>,0.330693
4,96,Dragonfly Coffee Roasters,Finca La Aurora Camilina Geisha,"Boulder, Colorado","Piedra Candela, Chiriqui Province, far western Panama",Medium-Light,2018-10-01,"Crisp, elegantly sweet, rich-toned. Dried mango, hazelnut butter, hibiscus, fresh-cut cedar, brown sugar in aroma and cup. Deeply sweet-tart structure with vibrant acidity; viscous, satiny mouthfeel. The finish leads with dried mango and hazelnut in the short, hibiscus and cedar returning in the long.","Tropical fruit- and spice-toned aromatic wood characterize this uniquely sweet, engagingly tart natural-processed Geisha cup.Explore Similar CoffeesClick here for more reviews from Dragonfly Coffee RoastersClick here for more information about coffees from Panama",56.0,76.0,<NA>,0.198416


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,NT_price,category,1763.0,74.42,1.0,NT
1,roast,category,11.0,0.46,5.0,"Medium-Light, Light, Medium, Medium-Dark, Dark"
2,review_date,datetime64[ns],0.0,0.00,61.0,"2022-11-01 00:00:00, 2023-03-01 00:00:00, 2022-10-01 00:00:00, 2021-11-01 00:00:00, 2021-03-01 00:00:00, 2021-08-01 00:00:00, 2020-08-01 00:00:00, 2020-10-01 00:00:00, 2021-04-01 00:00:00, 2022-02-01 00:00:00"
3,agtron_lower,float64,7.0,0.30,49.0,"58.0, 60.0, 56.0, 62.0, 54.0, 52.0, 64.0, 57.0, 59.0, 50.0"
4,agtron_upper,float64,7.0,0.30,54.0,"78.0, 76.0, 80.0, 74.0, 77.0, 72.0, 70.0, 82.0, 84.0, 68.0"
5,price_per_gram_in_usd,float64,0.0,0.00,656.0,"0.0529, 0.0588, 0.0559, 0.0647, 0.047, 0.0617, 0.05, 0.0735, 0.0676, 0.0705"
6,rating,int64,0.0,0.00,17.0,"93, 94, 92, 95, 91, 90, 96, 88, 97, 87"
7,desc_1,string,1.0,0.04,2367.0,"Richly chocolaty, deeply sweet. Dark chocolate, freesia-like flowers, crisp pear, vanilla bean, cedar in aroma and cup. Sweet in structure with gentle acidity; velvety-smooth mouthfeel. Quiet, chocolaty finish with notes of pear and cedar., Crisply sweet, balanced. Nectarine, caramelized almonds, freesia-like flowers, baking chocolate, cedar in aroma and cup. Sweet structure with bright, juicy acidity; velvety mouthfeel. Almond and chocolate-toned finish., Richly chocolaty, floral-toned. Chocolate fudge, dried black cherry, roasted almond butter, narcissus, marjoram in aroma and cup. Deeply sweet structure with gentle but harmonious acidity; creamy, viscous mouthfeel. Chocolaty and richly floral finish, long and lingering., Sweet-toned, gently fermenty. Lime zest, magnolia, cocoa nib, watermelon candy, a hint of aged grappa in aroma and cup. Brightly sweet with high-toned acidity; full, syrupy-smooth mouthfeel. In the short finish, crisp cocoa nib gives weight to watermelon candy notes; lime zest resurfaces in the long. , Deeply fruit-toned, balanced and deep. Dark chocolate, goji berry, gardenia, hazelnut butter, fresh-cut cedar in aroma and cup. Richly sweet structure with gentle, balanced acidity; crisp, syrupy mouthfeel Resonantly chocolaty finish with undertones of goji berry and hazelnut., Decadently rich, resonantly chocolaty, floral-toned. Chocolate fudge, salted caramel, gardenia, date, scorched almond in aroma and cup. Vibrantly bittersweet structure with round acidity; lush, creamy mouthfeel. Chocolaty and nut-toned finish with a hint of smokiness., Crisply sweet, bright and balanced. Cocoa nib, cherry, brown sugar, lemon thyme, magnolia in aroma and cup. Sweetly tart with brisk acidity; plush, syrupy mouthfeel. Cocoa-toned finish with notes of brown sugar and magnolia., Crisply chocolaty, nut-toned. Baking chocolate, pink grapefruit zest, hazelnut brittle, thyme, agave syrup in aroma and cup. Bittersweet structure with brisk acidity; smooth, satiny mouthfeel. Quiet, nut-driven finish with hints of baking chocolate., Richly sweet, deeply savory. Fig, dark chocolate, magnolia, cashew butter, a hint of grappa barrel in aroma and cup. Sweet-toned structure with juicy acidity; full, syrupy-smooth mouthfeel. The finish centers around notes of dark chocolate and cashew butter with undertones of dried fig., Sweet-savory, richly aromatic. Dried apricot, young goat cheese, dark chocolate, lemon-thyme, sandalwood in aroma and cup. Savory-leaning structure with sweet-tart acidity; syrupy-smooth mouthfeel. Fruit-toned, chocolaty, tangy finish."
8,desc_3,string,1.0,0.04,2368.0,"Pleasing aromatic cedar notes frame this fruit-driven blend of coffees from Ethiopia: berry-forward and gently spice-toned., The bottom line: A balanced, juicy, spice-toned Kenya with classic lead notes of currant and bittersweet citrus undertones., A berry-driven natural-processed Ethiopia — think blueberry, boysenberry, blackberry, raspberry all in one — supported by crisp chocolate and spicy florals., A balanced, nuanced washed Ethiopia cup with pretty citrus and sweet herb notes throughout, supported by cocoa tones.,

In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
rating,2369.0,93.071338,1.658687,82.000000,98.0
agtron_lower,2362.0,56.525402,5.948137,0.000000,85.0
agtron_upper,2362.0,74.726926,7.326964,0.000000,105.0
price_per_gram_in_usd,2369.0,0.910395,5.530759,0.001152,250.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  \
column      rank                                                                        
NT_price    1                                                                    <NA>   
            2                                                                      NT   
desc_1      1     Richly chocolaty, deeply sweet. Dark chocolate, freesia-like flo...   
            2     Crisply sweet, balanced. Nectarine, caramelized almonds, freesia...   
            3     Richly chocolaty, floral-toned. Chocolate fudge, dried black che...   
            4     Sweet-toned, gently fermenty. Lime zest, magnolia, cocoa nib, wa...   
            5     Deeply fruit-toned, balanced and deep. Dark chocolate, goji berr...   
desc_3      1     A deep-toned, sweetly savory Kenya cup with inviting floral note...   
            2     Pleasing aromatic cedar notes frame this fruit-driven blend of c...   
            3     The bottom line: A balanced, juicy, spice-toned Kenya with class...   
            4     A berry-driven natural-processed Ethiopia — think blueberry, boy...   
            5     A balanced, nuanced washed Ethiopia cup with pretty citrus and s...   
location    1                                                      Madison, Wisconsin   
            2                                                         Chia-Yi, Taiwan   
            3                                                          Taipei, Taiwan   
            4                                                   San Diego, California   
            5                                                  Minneapolis, Minnesota   
name        1                                                          Espresso Blend   
            2                                                   Colombia Pink Bourbon   
            3                                                 Bella Carmona Guatemala   
            4                                                           Holiday Blend   
            5                                                 Ethiopia Kayon Mountain   
origin      1                           Yirgacheffe growing region, southern Ethiopia   
            2                             Guji Zone, Oromia Region, southern Ethiopia   
            3                               Nyeri growing region, south-central Kenya   
            4                                  Boquete growing region, western Panama   
            5             Sidamo (also Sidama) growing region, south-central Ethiopia   
review_date 1                                                     2022-11-01 00:00:00   
            2                                                     2023-03-01 00:00:00   
            3                                                     2022-10-01 00:00:00   
            4                                                     2021-11-01 00:00:00   
            5                                                     2021-03-01 00:00:00   
roast       1                                                            Medium-Light   
            2                                                                   Light   
            3                                                                  Medium   
            4                                                             Medium-Dark   
            5                                                                    <NA>   
roaster     1                                                     JBC Coffee Roasters   
            2                                                       Paradise Roasters   
            3                                                           Kakalove Cafe   
            4                                                RamsHead Coffee Roasters   
            5                                                  Temple Coffee Roasters   

                  count    pct  
column      rank                
NT_price    1      1763  74.42  
            2       606  25.58  
desc_1      1         2   0.08 

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-1.145,-1.279,2.751,0.0,log,51223.9,8.244592e+16,exponential


## Task Curation

In [9]:
from data_foundry.schema import PredictiveMLSplitsMetadata

splits = {}

date_col = task_mold.time_on
step_months = 6
test_months = 6
steps_back = 5

max_month = df[date_col].max().to_period("M").to_timestamp()
test_end = (max_month + pd.offsets.MonthBegin(1))

for i in range(steps_back):
    test_start = test_end - pd.DateOffset(months=test_months)

    train_mask = df[date_col] < test_start
    test_mask  = (df[date_col] >= test_start) & (df[date_col] < test_end)

    train_idx = list(df.index[train_mask].to_numpy())
    test_idx  = list(df.index[test_mask].to_numpy())

    print(len(train_idx), len(test_idx))
    assert df[date_col].iloc[train_idx].max() < df[date_col].iloc[test_idx].min(), "Train and test sets are not properly separated in time."
    splits[i] = {0: (train_idx, test_idx)}

    test_end = test_end - pd.DateOffset(months=step_months)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We simulate a setting where we refit the model every 6 months and only test on the next 6 months. We repeat this process 9 times, going back in time. Note, our training data gets smaller step by step, down to 1k samples.",
    splits=splits,
    time_horizon=6,
    time_horizon_unit="months",
)

2137 232
1868 269
1581 287
1302 279


1045 257


"## Export

In [10]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to coffee_rating_prediction/019d7388-dffd-7bc4-ab3d-40bf6581786e
019d7388-dffd-7bc4-ab3d-40bf6581786e
fc00818a00199948c17196986d05c37e705f4cda5c5f7519511b929b756774da
